# ModelFlow in the browser

Python runs inside your browser (Pyodide); nothing is installed on your computer.
The first code cell installs ModelFlow into the browser session. It takes a little while the
first time and has to be run again each time the page is reloaded.

In JupyterLite press **Run > Run All Cells** in the menu line. In the book, start Python with
the power button and run the cells from the top.

## Now import ModelFlow

In [25]:
import sys
import importlib.util

if 'google.colab' in str(get_ipython()):
    if importlib.util.find_spec("modelclass") is None:
        !apt-get -qq install -y graphviz
        !pip -qq install ModelFlowIb > /dev/null 2>&1
        !mkdir -p data && curl -L https://raw.githubusercontent.com/IbHansen/modelflow-manual/main/model_repo/pak.pcim -o data/pak.pcim



if sys.platform == 'emscripten':  # in the browser: JupyterLite or the book's live code (thebe-lite)
    import micropip
    await micropip.install('modelflowib>=2.85')  # 2.85: no pyarrow in the browser, which the book's Pyodide lacks
    # the book's live code has no data folder, so fetch the model file into one
    from pathlib import Path
    from pyodide.http import pyfetch
    if not Path('data/pak.pcim').exists():
        Path('data').mkdir(exist_ok=True)
        response = await pyfetch('https://raw.githubusercontent.com/IbHansen/modelflow-manual/main/model_repo/pak.pcim')
        Path('data/pak.pcim').write_bytes(await response.bytes())

In [26]:
from modelclass import model
model.scroll_off() 
model.widescreen() 
model.plain_html=False 

<IPython.core.display.Javascript object>

## Load the Pakistan model and run the baseline

In [27]:
mpak,bline = model.modelload('data/pak.pcim',run=True,keep='Baseline')

Open file from URL:  https://raw.githubusercontent.com/IbHansen/modelflow-manual/main/model_repo/pak.pcim


Now we got a model `mpak` and a dataframe with a baseline `bline` 

A **dataframe** is a matix of all variable in the model with **variables** as columns and **years** as rows.

So we got all which is needed for creating experiments with the model. 

## Simulating the impact of imposing a carbon price

A simulation follows three steps:

1. **Create a scenario DataFrame** from a baseline using `.upd()`, which applies an update expression that updates selected variables over a chosen time range.
2. **Solve the model** by calling it with the scenario DataFrame, a start and end year, and a `keep=` label that names the scenario for later comparison.
3. **Inspect results** on the returned DataFrame or directly on the model, which stores each kept scenario.

### Create a scenario Dataframe

In [28]:
CT30df = bline.upd("<2025 2050> PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER = 30")

**Breakdown** 


$$
\begin{aligned}
&\underbrace{\texttt{CT30df}}_{\text{scenario dataframe}}
 = \underbrace{\texttt{bline}}_{\text{baseline dataframe}}
 \,.\,\underbrace{\texttt{upd}}_{\text{accessor}}\bigl(\texttt{"}\ldots\texttt{"}\bigr) \\[8pt]
&\text{where the update string is} \\[4pt]
&\texttt{"}\,
 \overbrace{\texttt{<2025 2050>}}^{\text{years}}\ 
 \overbrace{\texttt{PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER}}^{\text{coal, gas, oil CO}_2\text{ tax rates}}\ 
 \overbrace{\texttt{=}}^{\text{set}}\ 
 \overbrace{\texttt{30}}^{\text{USD/tCO}_2}
 \,\texttt{"}
\end{aligned}
$$

### Solve the model

In [29]:
resultsdf = mpak(CT30df,2020,2050,keep="Nominal $30 USD Carbon tax")

**Breakdown**
$$
\small
\underbrace{\texttt{resultsdf}}_{\text{results df}}
= \underbrace{\texttt{mpak}}_{\text{model}}\bigl(
\underbrace{\texttt{CT30df}}_{\text{scenario df}},\;
\underbrace{\texttt{2020}}_{\text{start}},\;
\underbrace{\texttt{2050}}_{\text{end}},\;
\underbrace{\texttt{keep="Nominal \$30USD Carbon tax"}}_{\text{scenario label}}
\bigr)
$$

where **df** is short for **dataframe** 

### Inspect results

 **Note**
 
 `resultsdf` is rarely used directly. The model object `mpak` keeps the baseline (`mpak.basedf`) and the latest run (`mpak.lastdf`) internally, so the following cells can compare and plot results directly from `mpak`.

 In the widget below the result for the variables `PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN` can be inspected. Try selecting the different tabs. 

In [30]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN']

### Show impact as nice graphs

In [31]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN'].rplot(samefig=0,datatype='difpctlevel',tab=True)

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

### How does the different variables influence the result of a endogenous variable 
In this case the GDP, but any endogenous variable can be shown 

In [32]:
mpak.PAKNYGDPMKTPKN.tracepre() 

#### The graph (network) can be pruned to show only variables with a max impact higher than a certain threshold

In [33]:
mpak.PAKNYGDPMKTPKN.tracepre(filter=20 ) 

#### and we can go several levels upstream
The filter=20 prunes the graph as it can be large

In [34]:
mpak.PAKNYGDPMKTPKN.tracepre(filter=20,up=2) 

### Attribution 
Try select different years


In [35]:
mpak.get_dekom_gui('PAKNYGDPMKTPKN')

interactive(children=(Dropdown(description='Variable', index=321, options=('CHNEXR05', 'CHNPCEXN05', 'DEUEXR05…

## Try some of the other notebooks in this example
Below a list of  notebooks which can be started by clicking. 

In [36]:
mpak.display_toc(text='**Try one of these notebooks**') 

**Try one of these notebooks**

interactive,ModelFlow input widgets
PAK_CarbonTax-Copy1,Now import ModelFlow
PAK_CarbonTax,This Notebook gives a short example of a policy simulation
pakstart,ModelFlow in the browser
rise test,
